<img src='http://hilpisch.com/taim_logo.png' width="350px" align="right">

# Artificial Intelligence in Finance

## Reinforcement Learning

&copy; Dr Yves J Hilpisch | The Python Quants GmbH

http://aimachine.io | http://twitter.com/dyjh

## Imports

In [ ]:
!git clone https://github.com/tpq-classes/ai_in_finance_book.git
import sys
sys.path.append('ai_in_finance_book')


In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
from pylab import plt, mpl
plt.style.use('seaborn-v0_8')
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'
np.set_printoptions(precision=4, suppress=True)
os.environ['PYTHONHASHSEED'] = '0'

## `CartPole` Environment

In [ ]:
import gymnasium as gym

In [ ]:
env = gym.make('CartPole-v1')

In [ ]:
env.action_space.seed(100)

In [ ]:
env.observation_space

In [ ]:
env.observation_space.low.astype(np.float16)

In [ ]:
env.observation_space.high.astype(np.float16)

In [ ]:
state = env.reset()

In [ ]:
state

In [ ]:
env.action_space

In [ ]:
env.action_space.n

In [ ]:
env.action_space.sample()

In [ ]:
env.action_space.sample()

In [ ]:
a = env.action_space.sample()
a

In [ ]:
state, reward, done, trunc, info = env.step(a)
state, reward, done, trunc, info

In [ ]:
env.reset()
for e in range(1, 200):
    a = env.action_space.sample()
    state, reward, done, trunc, info = env.step(a)
    print(f'step={e:2d} | state={state} | action={a} | reward={reward}')
    if done and (e + 1) < 200:
        print('*** FAILED ***')
        break

In [ ]:
done

## Dimensionality Reduction

See http://kvfrans.com/simple-algoritms-for-solving-cartpole/.

In [ ]:
np.random.seed(100)

In [ ]:
weights = np.random.random(4) * 2 - 1

In [ ]:
weights

In [ ]:
state, _ = env.reset()

In [ ]:
state

In [ ]:
s = np.dot(state, weights)
s

## Action Rule

In [ ]:
if s < 0:
    a = 0
else:
    a = 1

In [ ]:
a

## Total Reward per Episode

In [ ]:
def run_episode(env, weights):
    state,_  = env.reset()
    treward = 0
    for _ in range(200):
        s = np.dot(state, weights)
        a = 0 if s < 0 else 1
        state, reward, done, trunc, info = env.step(a)
        treward += reward
        if done:
            break
    return treward

In [ ]:
run_episode(env, weights)

## Simple Learning

In [ ]:
def set_seeds(seed=100):
    random.seed(seed)
    np.random.seed(seed)
    env.action_space.seed(seed)

In [ ]:
set_seeds()
num_episodes = 1000

In [ ]:
besttreward = 0
for e in range(1, num_episodes + 1):
    weights = np.random.rand(4) * 2 - 1
    treward = run_episode(env, weights)
    if treward > besttreward:
        besttreward = treward
        bestweights = weights
        if treward == 200:
            print(f'SUCCESS | episode={e}')
            break
        print(f'UPDATE  | episode={e}')

In [ ]:
weights

## Testing the Results

In [ ]:
res = []
for _ in range(100):
    treward = run_episode(env, weights)
    res.append(treward)
res[:10]

In [ ]:
sum(res) / len(res)

## DNN Learning

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '5'

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
from keras.layers import Dense, Dropout
from keras.models import Sequential
from sklearn.metrics import accuracy_score

In [ ]:
def set_seeds(seed=100):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    env.action_space.seed(seed)

In [ ]:
import tensorflow as tf

class NNAgent:
    def __init__(self):
        self.max = 0
        self.scores = []
        self.memory = []
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential([
            keras.Input(shape=(4,)),
            Dense(24, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(
            loss='binary_crossentropy',
            optimizer=keras.optimizers.RMSprop(
                learning_rate=0.001
            )
        )
        return model

    def act(self, state):
        if random.random() <= 0.5:
            return env.action_space.sample()

        prediction = self.model(state, training=False).numpy()[0, 0]
        return int(prediction > 0.5)

    def train_episode(self, episode_memory):
        if not episode_memory:
            return

        states = np.vstack([item[0] for item in episode_memory]).astype(np.float32)
        actions = np.array([item[1] for item in episode_memory], dtype=np.float32)
        self.model.train_on_batch(states, actions)

    def learn(self, episodes):
        for e in range(1, episodes + 1):
            state, _ = env.reset()
            episode_memory = []
            for step in range(201):
                state = np.asarray(state, dtype=np.float32).reshape(1, 4)
                action = self.act(state)
                next_state, reward, terminated, truncated, info = (env.step(action))
                done = bool(terminated) or bool(truncated)

                if done:
                    score = step + 1
                    self.scores.append(score)
                    self.max = max(score, self.max)
                    print(
                        f'episode: {e:4d}/{episodes} | '
                        f'score: {score:3d} | '
                        f'max: {self.max:3d}',
                        end='\r'
                    )
                    break

                experience = (state.copy(), action)

                # Permanent memory for later analysis
                self.memory.append(experience)

                # Temporary memory for this training batch
                episode_memory.append(experience)
                state = next_state

            # Train once on the complete episode
            self.train_episode(episode_memory)

        print()

In [ ]:
set_seeds(100)
agent = NNAgent()

In [ ]:
episodes = 1000

In [ ]:
agent.learn(episodes)

In [ ]:
sum(agent.scores) / len(agent.scores)

In [ ]:
f = np.array([m[0][0] for m in agent.memory])
f

In [ ]:
l = np.array([m[1] for m in agent.memory])
l

In [ ]:
accuracy_score(np.where(agent.model.predict(f) > 0.5, 1, 0), l)

## Q Learning

See https://keon.io/deep-q-learning/

In [ ]:
from collections import deque

In [ ]:
class DQLAgent:
    def __init__(self, gamma=0.95, hu=24,
                 opt=keras.optimizers.Adam,
                 lr=0.001, finish=False):

        self.finish = finish
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.gamma = gamma
        self.batch_size = 32

        self.max_treward = 0
        self.averages = []
        self.memory = deque(maxlen=2000)

        self.osn = env.observation_space.shape[0]
        self.n_actions = int(env.action_space.n)

        self.model = self._build_model(hu, opt, lr)
        self._update_weights()

    def _build_model(self, hu, opt, lr):
        model = Sequential([
            keras.Input(shape=(self.osn,)),
            Dense(hu, activation='relu'),
            Dense(hu, activation='relu'),
            Dense(self.n_actions, activation='linear')
        ])

        model.compile(loss='mse', optimizer=opt(learning_rate=lr))
        return model

    def _update_weights(self):
        self.w1, self.b1 = (self.model.layers[0].get_weights())
        self.w2, self.b2 = (self.model.layers[1].get_weights())
        self.w3, self.b3 = (self.model.layers[2].get_weights())

    def _predict_numpy(self, states):
        x = states @ self.w1 + self.b1
        x = np.maximum(x, 0)

        x = x @ self.w2 + self.b2
        x = np.maximum(x, 0)

        return x @ self.w3 + self.b3

    def act(self, state):
        if random.random() <= self.epsilon:
            return env.action_space.sample()

        values = self._predict_numpy(state)[0]
        return int(np.argmax(values))

    def replay(self):
        batch = random.sample(self.memory, self.batch_size)

        states = np.vstack([item[0] for item in batch]).astype(np.float32)
        actions = np.array([item[1] for item in batch], dtype=np.int32)
        rewards = np.array([item[2] for item in batch], dtype=np.float32)
        next_states = np.vstack([item[3] for item in batch]).astype(np.float32)
        dones = np.array([item[4] for item in batch], dtype=bool)

        # Fast NumPy inference for complete batches
        targets = self._predict_numpy(states)
        next_values = self._predict_numpy(next_states)
        future_rewards = np.max(next_values, axis=1)

        target_values = (rewards + (~dones) * self.gamma * future_rewards)
        targets[np.arange(self.batch_size), actions] = target_values

        # One TensorFlow training operation
        self.model.train_on_batch(states, targets)

        # Copy new weights back to NumPy
        self._update_weights()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= (self.epsilon_decay)

    def learn(self, episodes):
        trewards = []
        for e in range(1, episodes + 1):
            state, _ = env.reset()
            state = np.asarray(state, dtype=np.float32).reshape(1, self.osn)
            for step in range(5000):
                action = self.act(state)
                next_state, reward, terminated, truncated, info = (env.step(action))

                done = bool(terminated) or bool(truncated)
                next_state = np.asarray(next_state, dtype=np.float32
                                       ).reshape(1, self.osn)

                self.memory.append((
                    state,
                    action,
                    reward,
                    next_state,
                    done
                ))

                state = next_state
                if done:
                    treward = step + 1
                    trewards.append(treward)
                    av = np.mean(trewards[-25:])
                    self.averages.append(av)
                    self.max_treward = max(self.max_treward,treward)

                    print(
                        f'episode: {e:4d}/{episodes} | '
                        f'treward: {treward:4d} | '
                        f'av: {av:6.1f} | '
                        f'max: {self.max_treward:4d}',
                        end='\r'
                    )

                    break

            if (self.finish and len(trewards) >= 25 and av > 195):
                print()
                break

            if len(self.memory) >= self.batch_size:
                self.replay()

        print()

    def test(self, episodes):
        trewards = []
        for e in range(1, episodes + 1):
            state, _ = env.reset()
            for step in range(5001):
                state = np.asarray(state, dtype=np.float32
                                   ).reshape(1, self.osn)

                values = self._predict_numpy(state)[0]

                action = int(np.argmax(values))

                state, reward, terminated, truncated, info = (env.step(action))

                done = bool(terminated) or bool(truncated)
                if done:
                    treward = step + 1
                    trewards.append(treward)

                    print(
                        f'episode: {e:4d}/{episodes} | '
                        f'treward: {treward:4d}',
                        end='\r'
                    )

                    break
        print()
        return trewards

In [ ]:
episodes = 1000

In [ ]:
set_seeds(100)
agent = DQLAgent(finish=True)

In [ ]:
%time agent.learn(episodes)

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(agent.averages))
y = np.polyval(np.polyfit(x, agent.averages, deg=3), x)
plt.plot(agent.averages, label='moving average')
plt.plot(x, y, 'r--', label='trend')
plt.xlabel('episodes')
plt.ylabel('total reward')
plt.legend();

In [ ]:
trewards = agent.test(100)

In [ ]:
sum(trewards) / len(trewards)

## Finance Environment

In [ ]:
class observation_space:
    def __init__(self, n):
        self.shape = (n,)

In [ ]:
class action_space:
    def __init__(self, n):
        self.n = n
    def seed(self, seed):
        pass
    def sample(self):
        return random.randint(0, self.n - 1)

In [ ]:
class Finance:
    url = 'http://hilpisch.com/aiif_eikon_eod_data.csv'
    def __init__(self, symbol, features):
        self.symbol = symbol
        self.features = features
        self.observation_space = observation_space(4)
        self.osn = self.observation_space.shape[0]
        self.action_space = action_space(2)
        self.min_accuracy = 0.475
        self._get_data()
        self._prepare_data()

    def _get_data(self):
        self.raw = pd.read_csv(self.url, index_col=0,
                               parse_dates=True).dropna()
    def _prepare_data(self):
        self.data = pd.DataFrame(self.raw[self.symbol])
        self.data['r'] = np.log(self.data / self.data.shift(1))
        self.data.dropna(inplace=True)
        self.data = (self.data - self.data.mean()) / self.data.std()
        self.data['d'] = np.where(self.data['r'] > 0, 1, 0)

    def _get_state(self):
        return self.data[self.features].iloc[
            self.bar - self.osn:self.bar].values, None

    def seed(self, seed=None):
        pass

    def reset(self):
        self.treward = 0
        self.accuracy = 0
        self.bar = self.osn
        state = self.data[self.features].iloc[
            self.bar - self.osn:self.bar]
        return state.values, None

    def step(self, action):
        correct = action == self.data['d'].iloc[self.bar]
        reward = 1 if correct else 0
        self.treward += reward
        self.bar += 1
        self.accuracy = self.treward / (self.bar - self.osn)
        if self.bar >= len(self.data):
            done = True
        elif reward == 1:
            done = False
        elif (self.accuracy < self.min_accuracy and
              self.bar > self.osn + 10):
            done = True
        else:
            done = False
        state, _ = self._get_state()
        info = {}
        return state, reward, done, None, info

In [ ]:
env = Finance('EUR=', 'EUR=')

In [ ]:
env.reset()

In [ ]:
a = env.action_space.sample()
a

In [ ]:
env.step(a)

In [ ]:
set_seeds(100)
agent = DQLAgent(gamma=0.5, opt=keras.optimizers.RMSprop)

In [ ]:
episodes = 1000

In [ ]:
%time agent.learn(episodes)

In [ ]:
agent.test(3)

In [ ]:
plt.figure(figsize=(10, 6))
x = range(len(agent.averages))
y = np.polyval(np.polyfit(x, agent.averages, deg=3), x)
plt.plot(agent.averages, label='moving average')
plt.plot(x, y, 'r--', label='regression')
plt.xlabel('episodes')
plt.ylabel('total reward')
plt.legend();

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>